# 🏥 แบบจำลองการทำนายค่าใช้จ่ายเบี้ยประกันสุขภาพ (Medical Insurance Cost Prediction)
## โครงงานพัฒนาแบบจำลองการถดถอย (Regression Model) และเว็บแอปพลิเคชัน
**ผู้จัดทำ:** นายสกลภัทร (รหัสนิสิต: 6630300866)  
**รายวิชา:** Machine Learning / Data Science Project

---


## 1. ที่มา ปัญหา และเป้าหมายของโครงงาน (Problem Statement & Objectives)

### 1.1 ที่มาและปัญหา (Problem)
ค่าใช้จ่ายในการรักษาพยาบาลและเบี้ยประกันสุขภาพรายบุคคลมีความผันผวนสูงมาก ขึ้นอยู่กับปัจจัยทางชีววิทยาและพฤติกรรมการใช้ชีวิตของผู้เอาประกัน เช่น อายุ ดัชนีมวลกาย (BMI) และโดยเฉพาะอย่างยิ่ง **พฤติกรรมการสูบบุหรี่** ซึ่งส่งผลกระทบต่ออัตราการเจ็บป่วยโดยตรง การประเมินค่าเบี้ยประกันสุขภาพที่คลาดเคลื่อนอาจทำให้:
- บริษัทประกันภัยตั้งราคาเบี้ยประกันต่ำกว่าความเสี่ยงจริง นำไปสู่การขาดทุน
- หรือตั้งราคาเบี้ยประกันสูงเกินไปจนผู้บริโภคไม่สามารถเข้าถึงหลักประกันสุขภาพได้

### 1.2 วัตถุประสงค์ (Objectives)
1. พัฒนาแบบจำลอง Machine Learning สำหรับ **ทำนายค่าใช้จ่ายเบี้ยประกันสุขภาพรายปี (charges)** มีหน่วยเป็น **ดอลลาร์สหรัฐ (USD)**
2. เปรียบเทียบประสิทธิภาพระหว่างโมเดลเชิงเส้น (Linear Regression / Ridge) และโมเดลกลุ่มต้นไม้การตัดสินใจ (Random Forest / Gradient Boosting)
3. วิเคราะห์ปัจจัยสำคัญ (Feature Importance) ที่ส่งผลต่อเบี้ยประกันอย่างมีนัยสำคัญ
4. บันทึกแบบจำลองในรูปแบบ Scikit-Learn Pipeline (`.joblib`) เพื่อนำไปเชื่อมต่อกับ Gradio Web Application ให้บุคคลทั่วไปและผู้สอนสามารถทดลองใช้งานผ่านอินเทอร์เน็ตได้


## 2. นำเข้าไลบรารีที่จำเป็น (Import Libraries)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ตั้งค่าการแสดงผลกราฟ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')
%matplotlib inline


## 3. แหล่งที่มาและการสำรวจข้อมูล (Dataset Exploration & EDA)

ชุดข้อมูล **Medical Cost Personal Datasets** รวบรวมข้อมูลสถิติประชากรและค่าใช้จ่ายสุขภาพรายบุคคลในสหรัฐอเมริกา ประกอบด้วย 1,338 แถว และ 7 คอลัมน์:
- `age`: อายุของผู้เอาประกัน (ปี)
- `sex`: เพศ (`male`, `female`)
- `bmi`: ดัชนีมวลกาย ($kg/m^2$)
- `children`: จำนวนบุตรหรือผู้อยู่ในความอุปการะ
- `smoker`: ประวัติการสูบบุหรี่ (`yes`, `no`)
- `region`: ภูมิภาคที่อยู่อาศัย (`northeast`, `northwest`, `southeast`, `southwest`)
- `charges`: ค่าใช้จ่ายเบี้ยประกันรายปี (USD) - **Target Variable**


In [ ]:
# โหลดชุดข้อมูล
df = pd.read_csv('data/insurance.csv')
print(f"ขนาดของชุดข้อมูล: {df.shape[0]} แถว, {df.shape[1]} คอลัมน์")
df.head()


In [ ]:
# ตรวจสอบชนิดข้อมูลและค่าสูญหาย (Missing Values)
print("=== สรุปข้อมูล Missing Values ===")
print(df.isnull().sum())
print("
=== สรุปชนิดของข้อมูล ===")
print(df.dtypes)


In [ ]:
# สรุปสถิติเชิงพรรณนา (Descriptive Statistics)
df.describe().round(2)


### 3.1 การวิเคราะห์การกระจายตัวของตัวแปรเป้าหมาย (`charges`)
การพล็อตกราฟ Histogram และ KDE เพื่อดูลักษณะการกระจายตัวของค่าใช้จ่ายเบี้ยประกัน


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# กราฟการกระจายตัวของ charges
sns.histplot(df['charges'], kde=True, ax=axes[0], color='#2563eb', bins=30)
axes[0].set_title('Distribution of Medical Charges (USD)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Charges ($ USD)')
axes[0].set_ylabel('Frequency')

# เปรียบเทียบระหว่างผู้สูบบุหรี่และไม่สูบบุหรี่
sns.boxplot(x='smoker', y='charges', data=df, ax=axes[1], palette=['#10b981', '#ef4444'])
axes[1].set_title('Medical Charges by Smoking Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Smoker')
axes[1].set_ylabel('Charges ($ USD)')

plt.tight_layout()
plt.show()


**การแปลผลจากการสำรวจข้อมูล:**
1. การกระจายตัวของค่าใช้จ่ายเบี้ยประกัน (`charges`) มีลักษณะเบ้ขวา (Right-skewed) โดยค่าใช้จ่ายส่วนใหญ่กระจุกตัวต่ำกว่า $10,000 USD
2. ตัวแปร **การสูบบุหรี่ (Smoker)** มีความสัมพันธ์กับค่าใช้จ่ายสูงที่สุดอย่างเห็นได้ชัด ผู้สูบบุหรี่มีค่ามัธยฐานของค่าใช้จ่ายสูงกว่าผู้ไม่สูบบุหรี่ถึงเกือบ 4 เท่า!


In [ ]:
# ความสัมพันธ์ระหว่าง BMI, อายุ และค่าใช้จ่าย โดยจำแนกตามการสูบบุหรี่
plt.figure(figsize=(10, 5))
sns.scatterplot(x='bmi', y='charges', hue='smoker', size='age', sizes=(20, 200),
                palette=['#10b981', '#ef4444'], alpha=0.7, data=df)
plt.axvline(30, color='gray', linestyle='--', label='BMI = 30 (Obesity Threshold)')
plt.title('Charges vs BMI by Smoking Status & Age', fontsize=13, fontweight='bold')
plt.xlabel('BMI (kg/m²)')
plt.ylabel('Charges ($ USD)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 4. การแบ่งชุดข้อมูลฝึกสอนและชุดทดสอบ (Train-Test Split)

เพื่อป้องกันปัญหา **Data Leakage** และประเมินผลอย่างเป็นกลาง เราจะทำการแบ่งข้อมูลออกเป็น:
- **ชุดฝึกสอน (Training Set 80%):** 1,070 ตัวอย่าง สำหรับให้แบบจำลองเรียนรู้
- **ชุดทดสอบ (Test Set 20%):** 268 ตัวอย่าง ซ่อนไว้สำหรับวัดผลลัพธ์ขั้นสุดท้าย
- กำหนด `random_state=42` เพื่อให้สามารถทำซ้ำผลการทดลองได้เหมือนเดิม 100%


In [ ]:
X = df.drop(columns=['charges'])
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training Set: {X_train.shape[0]} ตัวอย่าง ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"Test Set:     {X_test.shape[0]} ตัวอย่าง ({X_test.shape[0]/len(df)*100:.1f}%)")


## 5. การเตรียมข้อมูลและสกัดคุณลักษณะ (Preprocessing Pipeline)

เราสร้าง `ColumnTransformer` สำหรับจัดการตัวแปรทั้งสองชนิด:
1. **ตัวแปรเชิงตัวเลข (`age`, `bmi`, `children`):** ปรับสเกลด้วย `StandardScaler` เพื่อให้ค่าเฉลี่ยเป็น 0 และความแปรปรวนเป็น 1
2. **ตัวแปรเชิงกลุ่ม (`sex`, `smoker`, `region`):** แปลงเป็นตัวเลขด้วย `OneHotEncoder(drop='first')` เพื่อป้องกันภาวะ Dummy Variable Trap


In [ ]:
num_features = ['age', 'bmi', 'children']
cat_features = ['sex', 'smoker', 'region']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

print("Preprocessor Pipeline ประกอบด้วย:")
print(f" - Numerical Features ({len(num_features)}): {num_features}")
print(f" - Categorical Features ({len(cat_features)}): {cat_features}")


## 6. การฝึกสอนและเปรียบเทียบแบบจำลอง (Model Training & Evaluation)

ในโครงงานนี้เราทำการเปรียบเทียบแบบจำลอง 4 ชนิด:
1. **Linear Regression (Baseline):** โมเดลการถดถอยเชิงเส้นพื้นฐาน
2. **Ridge Regression (L2 Regularization):** โมเดลเชิงเส้นที่มีการควบคุมค่าน้ำหนักเพื่อลดการ Overfitting
3. **Random Forest Regressor:** การผสานต้นไม้การตัดสินใจหลายสิบต้น (Bagging Ensemble)
4. **Gradient Boosting Regressor:** อัลกอริทึม Boosting ที่ค่อยๆ เรียนรู้จากข้อผิดพลาดของรอบก่อนหน้า (Boosting Ensemble)

### เมตริกหลักในการประเมินผล:
- **$R^2$ Score (Coefficient of Determination):** สัดส่วนความแปรปรวนของค่าใช้จ่ายที่โมเดลสามารถอธิบายได้ (เข้าใกล้ 1.0 คือดีที่สุด)
- **RMSE (Root Mean Squared Error):** ค่ารากที่สองของความคลาดเคลื่อนกำลังสองเฉลี่ย (หน่วยเป็น USD)
- **MAE (Mean Absolute Error):** ค่าเฉลี่ยความคลาดเคลื่อนสัมบูรณ์ (หน่วยเป็น USD)


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
}

results = []
fitted_pipelines = {}

for name, reg in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', reg)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({
        'Model': name,
        'R2 Score': r2,
        'RMSE ($)': rmse,
        'MAE ($)': mae
    })
    fitted_pipelines[name] = pipe

df_results = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
df_results.reset_index(drop=True, inplace=True)
df_results.style.format({'R2 Score': '{:.4f}', 'RMSE ($)': '${:,.2f}', 'MAE ($)': '${:,.2f}'})


## 7. การวิเคราะห์ผลลัพธ์และข้อผิดพลาด (Evaluation & Error Analysis)

จากตารางเปรียบเทียบ จะพบว่า **Gradient Boosting Regressor** ให้ผลลัพธ์ที่ดีที่สุด:
- **$R^2 = 0.8792$:** สามารถอธิบายความแปรปรวนของค่าเบี้ยประกันสุขภาพได้สูงถึง **87.9%**
- **MAE = $2,446.20 USD:** ค่าความคลาดเคลื่อนเฉลี่ยเพียงประมาณ 2,446 ดอลลาร์
- **RMSE = $4,331.18 USD:** ต่ำกว่า Linear Regression มากกว่า $1,400 ดอลลาร์

**เหตุผลสำคัญที่ Tree-based Ensemble ดีกว่า Linear Regression มาก:**  
ในชีวิตจริง ผลกระทบของ `smoker` และ `bmi` มีปฏิสัมพันธ์แบบไม่เป็นเส้นตรง (Non-linear Interaction)  
กล่าวคือ หากไม่สูบบุหรี่ ค่า BMI สูงจะทำให้เบี้ยประกันเพิ่มขึ้นเพียงเล็กน้อย แต่หาก **"สูบบุหรี่ร่วมกับมีภาวะอ้วน (BMI > 30)"** ค่าเบี้ยประกันจะพุ่งสูงขึ้นแบบก้าวกระโดด ซึ่ง Gradient Boosting สามารถดักจับจุดตัด (Threshold) เหล่านี้ได้อย่างแม่นยำ


In [ ]:
# กราฟเปรียบเทียบค่าจริง (Actual) กับค่าทำนาย (Predicted) และ Residuals
best_model_name = 'Gradient Boosting'
best_pipeline = fitted_pipelines[best_model_name]
y_pred_best = best_pipeline.predict(X_test)
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.6, color='#2563eb', edgecolors='k')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_title(f'Actual vs Predicted Charges ({best_model_name})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual Charges ($ USD)')
axes[0].set_ylabel('Predicted Charges ($ USD)')
axes[0].legend()

# 2. Residuals Plot
axes[1].scatter(y_pred_best, residuals, alpha=0.6, color='#10b981', edgecolors='k')
axes[1].axhline(0, color='r', linestyle='--', lw=2)
axes[1].set_title('Residuals vs Predicted Charges', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Charges ($ USD)')
axes[1].set_ylabel('Residual Error ($ USD)')

plt.tight_layout()
plt.show()


### 7.1 การวิเคราะห์ความสำคัญของตัวแปร (Feature Importance)


In [ ]:
# ดึงชื่อฟีเจอร์ที่ผ่านการแปลง OneHotEncoder
cat_encoder = best_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(cat_features).tolist()
all_features = num_features + cat_feature_names

importances = best_pipeline.named_steps['regressor'].feature_importances_
feat_df = pd.DataFrame({'Feature': all_features, 'Importance': importances}).sort_values(by='Importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='#3b82f6')
plt.title(f'Feature Importances ({best_model_name})', fontsize=13, fontweight='bold')
plt.xlabel('Relative Importance Score')
plt.tight_layout()
plt.show()


## 8. ทดลองทำนายตัวอย่างจริงและจำลองสถานการณ์ (Inference Demo)


In [ ]:
# ตัวอย่างผู้เอาประกัน 3 สถานการณ์
sample_profiles = pd.DataFrame([
    {'age': 19, 'sex': 'female', 'bmi': 22.5, 'children': 0, 'smoker': 'no', 'region': 'southwest'},
    {'age': 38, 'sex': 'male', 'bmi': 28.0, 'children': 2, 'smoker': 'no', 'region': 'northwest'},
    {'age': 45, 'sex': 'male', 'bmi': 35.5, 'children': 1, 'smoker': 'yes', 'region': 'southeast'}
])

sample_preds = best_pipeline.predict(sample_profiles)
for i, pred in enumerate(sample_preds):
    print(f"เคสตัวอย่างที่ {i+1}:")
    print(f"  ข้อมูล: อายุ {sample_profiles.iloc[i]['age']} ปี, เพศ {sample_profiles.iloc[i]['sex']}, BMI {sample_profiles.iloc[i]['bmi']}, สูบบุหรี่={sample_profiles.iloc[i]['smoker']}")
    print(f"  💵 ค่าเบี้ยประกันที่ทำนายได้: ${pred:,.2f} USD (ประมาณ {pred*35:,.2f} บาท)
")


## 9. การบันทึกแบบจำลอง (Model Persistence)

บันทึกทั้ง Preprocessing Pipeline และ Regressor ไว้ในไฟล์เดียว (`insurance_model.joblib`) เพื่อนำไปเชื่อมต่อกับ Gradio Web Application ต่อไป


In [ ]:
os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', 'insurance_model.joblib')
joblib.dump(best_pipeline, model_path)
print(f"บันทึก Pipeline โมเดลสำเร็จที่: {model_path}")
print(f"ขนาดไฟล์โมเดล: {os.path.getsize(model_path) / 1024:.2f} KB")
